# 第8章: ニューラルネット

第7章で取り組んだポジネガ分類を題材として、ニューラルネットワークで分類モデルを実装する。なお、この章ではPyTorchやTensorFlow、JAXなどの深層学習フレームワークを活用せよ。

## 70. 単語埋め込みの読み込み

事前学習済み単語埋め込みを活用し、$|V| \times d_\rm{emb}$ の単語埋め込み行列$\pmb{E}$を作成せよ。ここで、$|V|$は単語埋め込みの語彙数、$d_\rm{emb}$は単語埋め込みの次元数である。ただし、単語埋め込み行列の先頭の行ベクトル$\pmb{E}_{0,:}$は、将来的にパディング（`<PAD>`）トークンの埋め込みベクトルとして用いたいので、ゼロベクトルとして予約せよ。ゆえに、$\pmb{E}$の2行目以降に事前学習済み単語埋め込みを読み込むことになる。

もし、Google Newsデータセットの[学習済み単語ベクトル](https://drive.google.com/file/d/0B7XkCwpI5KDYNlNUTTlSS21pQmM/edit?usp=sharing)（300万単語・フレーズ、300次元）を全て読み込んだ場合、$|V|=3000001, d_\rm{emb}=300$になるはずである（ただ、300万単語の中には、殆ど用いられない稀な単語も含まれるので、語彙を削減した方がメモリの節約になる）。

また、単語埋め込み行列の構築と同時に、単語埋め込み行列の各行のインデックス番号（トークンID）と、単語（トークン）への双方向の対応付けを保持せよ。

In [2]:
!pip install torch numpy gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 69.0 MB/s eta 0:00:00


In [3]:
import torch
import torch.nn as nn
import numpy as np
from gensim.models import KeyedVectors

def build_embedding_layer(w2v_path, max_vocab=None):
    """
    事前学習済みWord2Vecを読み込み、PyTorchのEmbedding層と双方向の辞書を作成する関数

    Args:
        w2v_path (str): 事前学習済みWord2Vecファイル（.binなど）のパス
        max_vocab (int, optional): 読み込む最大語彙数。指定すると頻出単語から順に読み込みメモリを節約する。

    Returns:
        embedding_layer (nn.Embedding): PyTorchの埋め込み層
        word_to_id (dict): 単語からIDへのマッピング
        id_to_word (dict): IDから単語へのマッピング
    """
    print("事前学習済み単語ベクトルを読み込んでいます...")
    # Gensimを使用してモデルを読み込む（limit引数で語彙数の削減が可能）
    wv = KeyedVectors.load_word2vec_format(w2v_path, binary=True, limit=max_vocab)
    d_emb = wv.vector_size

    # 双方向の対応付けと重みリストの初期化
    word_to_id = {}
    id_to_word = {}
    weights = []

    # 1. 行列の先頭 (インデックス0) に <PAD> トークンを予約（ゼロベクトル）
    pad_token = "<PAD>"
    word_to_id[pad_token] = 0
    id_to_word[0] = pad_token
    # E_{0,:} として d_emb 次元のゼロベクトルを追加
    weights.append(np.zeros(d_emb, dtype=np.float32))

    # 2. 事前学習済み単語をマッピングと重み行列 (2行目以降) に追加
    for i, word in enumerate(wv.index_to_key):
        token_id = i + 1  # 0は <PAD> のため1からインデックスを割り当て
        word_to_id[word] = token_id
        id_to_word[token_id] = word
        weights.append(wv[word])

    # 3. 重みリストをPyTorchのTensorに変換
    # 形状は |V| × d_emb になります (|V| は max_vocab + 1)
    weights_tensor = torch.FloatTensor(np.array(weights))

    # 4. PyTorchのEmbedding層の構築
    # padding_idx=0 を指定することで、モデルはID=0をパディングとして認識し、
    # ゼロベクトルのまま勾配計算（更新）の対象から外すことができます。
    embedding_layer = nn.Embedding.from_pretrained(
        weights_tensor,
        freeze=False,  # 学習中に単語ベクトルも更新（ファインチューニング）する場合はFalse、固定ならTrue
        padding_idx=0
    )

    return embedding_layer, word_to_id, id_to_word

# ==========================================
# 実行例
# ==========================================
if __name__ == "__main__":
    # Google Newsデータセットの想定パス（実際のパスに書き換えてください）
    w2v_file_path = "GoogleNews-vectors-negative300.bin"

    # メモリ節約のため、頻出上位100,000単語のみを読み込む設定
    try:
        embed_layer, w2id, id2w = build_embedding_layer(w2v_file_path, max_vocab=100000)

        # 構築結果の確認
        vocab_size = len(w2id)
        d_emb = embed_layer.embedding_dim
        print("\n--- 構築完了 ---")
        print(f"語彙数 |V| (<PAD>含む): {vocab_size}")
        print(f"埋め込み次元数 d_emb: {d_emb}")
        print(f"単語埋め込み行列 E の形状: {embed_layer.weight.shape}")

        # 動作確認: <PAD> (ID=0) がゼロベクトル E_{0,:} として保持されているか
        pad_vector = embed_layer(torch.tensor(0))
        print(f"\n<PAD> (ID=0) のベクトル (先頭5要素):\n{pad_vector[:5].detach().numpy()}")

        # 動作確認: 任意の単語からIDを取得し、埋め込みベクトルを取り出す
        sample_word = "apple"
        if sample_word in w2id:
            word_id = w2id[sample_word]
            word_vector = embed_layer(torch.tensor(word_id))
            print(f"\n単語 '{sample_word}' のID: {word_id}")
            print(f"単語 '{sample_word}' のベクトル (先頭5要素):\n{word_vector[:5].detach().numpy()}")

    except FileNotFoundError:
        print(f"エラー: {w2v_file_path} が見つかりません。実行する際は実際のファイルを用意してください。")

事前学習済み単語ベクトルを読み込んでいます...
エラー: GoogleNews-vectors-negative300.bin が見つかりません。実行する際は実際のファイルを用意してください。


## 71. データセットの読み込み

[General Language Understanding Evaluation (GLUE)](https://gluebenchmark.com/) ベンチマークで配布されている[Stanford Sentiment Treebank (SST)](https://dl.fbaipublicfiles.com/glue/data/SST-2.zip) をダウンロードし、訓練セット（train.tsv）と開発セット（dev.tsv）のテキストと極性ラベルと読み込み、全てのテキストをトークンID列に変換せよ。このとき、単語埋め込みの語彙でカバーされていない単語は無視し、トークン列に含めないことにせよ。また、テキストの全トークンが単語埋め込みの語彙に含まれておらず、空のトークン列となってしまう事例は、訓練セットおよび開発セットから削除せよ（このため、第7章の実験で得られた正解率と比較できなくなることに注意せよ）。

事例の表現方法は任意でよいが、例えば"contains no wit , only labored gags"がネガティブに分類される事例は、次のような辞書オブジェクトで表現すればよい。

```
{'text': 'contains no wit , only labored gags',
 'label': tensor([0.]),
 'input_ids': tensor([ 3475,    87, 15888,    90, 27695, 42637])}
```

この例では、`text`はテキスト、`label`は分類ラベル（ポジティブなら`tensor([1.])`、ネガティブなら`tensor([0.])`）、`input_ids`はテキストのトークン列をID列で表現している。

## 72. Bag of wordsモデルの構築

単語埋め込みの平均ベクトルでテキストの特徴ベクトルを表現し、重みベクトルとの内積でポジティブ及びネガティブを分類するニューラルネットワーク（ロジスティック回帰モデル）を設計せよ。

## 73. モデルの学習

問題72で設計したモデルの重みベクトルを訓練セット上で学習せよ。ただし、学習中は単語埋め込み行列の値を固定せよ（単語埋め込み行列のファインチューニングは行わない）。また、学習時に損失値を表示するなど、学習の進捗状況をモニタリングできるようにせよ。

## 74. モデルの評価

問題73で学習したモデルの開発セットにおける正解率を求めよ。

## 75. パディング

複数の事例が与えられたとき、これらをまとめて一つのテンソル・オブジェクトで表現する関数`collate`を実装せよ。与えられた複数の事例のトークン列の長さが異なるときは、トークン列の長さが最も長いものに揃え、0番のトークンIDでパディングをせよ。さらに、トークン列の長さが長いものから順に、事例を並び替えよ。

例えば、訓練データセットの冒頭の4事例が次のように表されているとき、

```
[{'text': 'hide new secretions from the parental units',
  'label': tensor([0.]),
  'input_ids': tensor([  5785,     66, 113845,     18,     12,  15095,   1594])},
 {'text': 'contains no wit , only labored gags',
  'label': tensor([0.]),
  'input_ids': tensor([ 3475,    87, 15888,    90, 27695, 42637])},
 {'text': 'that loves its characters and communicates something rather beautiful about human nature',
  'label': tensor([1.]),
  'input_ids': tensor([    4,  5053,    45,  3305, 31647,   348,   904,  2815,    47,  1276,  1964])},
 {'text': 'remains utterly satisfied to remain the same throughout',
  'label': tensor([0.]),
  'input_ids': tensor([  987, 14528,  4941,   873,    12,   208,   898])}]
```

`collate`関数を通した結果は以下のようになることが想定される。

```
{'input_ids': tensor([
    [     4,   5053,     45,   3305,  31647,    348,    904,   2815,     47,   1276,   1964],
    [  5785,     66, 113845,     18,     12,  15095,   1594,      0,      0,      0,      0],
    [   987,  14528,   4941,    873,     12,    208,    898,      0,      0,      0,      0],
    [  3475,     87,  15888,     90,  27695,  42637,      0,      0,      0,      0,      0]]),
 'label': tensor([
    [1.],
    [0.],
    [0.],
    [0.]])}
```


## 76. ミニバッチ学習

問題75のパディングの処理を活用して、ミニバッチでモデルを学習せよ。また、学習したモデルの開発セットにおける正解率を求めよ。

## 77. GPU上での学習

問題76のモデル学習をGPU上で実行せよ。また、学習したモデルの開発セットにおける正解率を求めよ。

## 78. 単語埋め込みのファインチューニング

問題77の学習において、単語埋め込みのパラメータも同時に更新するファインチューニングを導入せよ。また、学習したモデルの開発セットにおける正解率を求めよ。

## 79. アーキテクチャの変更

ニューラルネットワークのアーキテクチャを自由に変更し、モデルを学習せよ。また、学習したモデルの開発セットにおける正解率を求めよ。例えば、テキストの特徴ベクトル（単語埋め込みの平均ベクトル）に対して多層のニューラルネットワークを通したり、畳み込みニューラルネットワーク（CNN; Convolutional Neural Network）や再帰型ニューラルネットワーク（RNN; Recurrent Neural Network）などのモデルの学習に挑戦するとよい。